## POT RD, In-Situ, Time Varying

In this section, 
1. Time varying thresholds, POT of RD, and associated NTR and RF in a 2.5 days windows are achieved.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob
from datetime import timedelta
import os

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'dateTime'

# Name of variable column in each dataset
ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
acc_hr = 'acc_24hr'

events_per_year = 5 # Number of events to sample per year 

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate') # Directory that is used to save the POT CSVs for 
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets') # Directory where the original datasets are stored. This is used to read in the data and create the POT CSVs 

In [ ]:
for loc in locations:

    base_path = core_directory / Path(f'{loc}')

    # ────────────────────────────────────────────────────────────────────────────────
    # 📌 Load datasets
    # ────────────────────────────────────────────────────────────────────────────────
    df_rf = pd.read_csv(rf'{dataset_directory}/In_Situ/GHCN_Rainfall_1950_2050_hourly_accumulated.csv')
    df_rf['time'] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index('time')
    df_rf = df_rf[acc_hr]
    
    df_ntr = pd.read_csv(rf'{dataset_directory}/In_Situ/NOAA_Tide_Gauge_MSL_All_Predicted.csv')
    df_ntr['time'] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index('time')
    
    df_rd = pd.read_csv(rf'{dataset_directory}/In_Situ/usgs_discharge.csv')
    df_rd['time'] = pd.to_datetime(df_rd[time_col_rd])
    df_rd = df_rd.drop(columns=[time_col_rd])
    df_rd = df_rd.set_index('time')
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    rainfall_series = df_rf.dropna()
    decluster_window = timedelta(days=2.5)
    
    output_dir = rf'{base_path}/TimeVarying'
    
    rd_series = df_rd[rd_col].dropna()
    # ────────────────────────────────────────────────────────────────────────────────
    # Storage
    # ────────────────────────────────────────────────────────────────────────────────
    all_pot = []
    thresholds = []
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Yearly POT extraction + NTR matching
    # ────────────────────────────────────────────────────────────────────────────────
    for year, group in rd_series.groupby(rd_series.index.year):
        group_sorted = group.sort_values(ascending=False)
        selected = []
    
        for idx, val in group_sorted.items():
            if any(abs((idx - sel).days) < decluster_window.days for sel in selected):
                continue
            selected.append(idx)
            if len(selected) == events_per_year:
                break
    
        if len(selected) < events_per_year:
            continue
    
        # Minimum of selected events becomes the threshold for that year
        pot_values = [rd_series.loc[t] for t in selected]
        threshold_value = min(pot_values)
        thresholds.append({'Year': year, 'Threshold': threshold_value})
    
        # Match with NTR
        for t, val in zip(selected, pot_values):
            start = t - decluster_window
            end = t + decluster_window
            ntr_window = df_ntr.loc[start:end]
            max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan
            all_pot.append({'time': t, 'POT_RD': val, 'Max_NTR_in_5d': max_ntr, 'Year': year})
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Save outputs
    # ────────────────────────────────────────────────────────────────────────────────
    df_pot = pd.DataFrame(all_pot).sort_values('time').reset_index(drop=True)
    df_thresh = pd.DataFrame(thresholds)
    
    df_pot.to_csv(os.path.join(output_dir, f'POT_RD_Time_Varying_{events_per_year}.csv'), index=False)
    df_thresh.to_csv(os.path.join(output_dir, f'RD_TimeVarying_Thresholds_{events_per_year}.csv'), index=False)
    
    df_pot = pd.read_csv(os.path.join(output_dir, f'POT_RD_Time_Varying_{events_per_year}.csv'))
    df_pot = df_pot.dropna()
    df_pot = df_pot.drop(columns=['Year'])
    df_pot['time'] = pd.to_datetime(df_pot['time'])
    df_pot = df_pot.set_index('time')
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in df_pot.index:
        # event_date = pd.to_datetime(event_time)  # usually not necessary if already Timestamp
    
        start_date = event_time - timedelta(days=2.5)
        end_date = event_time + timedelta(days=2.5)
    
        # Subset rainfall data in that window
        window_data = df_rf.loc[start_date:end_date]
    
        # Get max rainfall values and store with event label
        max_value = window_data.max()
        max_rd_series = pd.Series([max_value], name=event_time)
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    df_pot.index = pd.to_datetime(df_pot.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    df_pot = df_pot.copy()
    df_pot = df_pot.reset_index().rename(columns={'time': 'date'})
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  
    
    # Step 3: Merge on 'date'
    merged_df = df_pot.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.rename(columns={'date':'time'})
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={0:'Max_RF_in_5d'})
    merged_df['time'] = pd.to_datetime(merged_df['time'])
                   
    merged_df.to_csv(rf'{output_dir}/POT_RD_TimeVarying_NTR_RF_{events_per_year}.csv',index=False)                
    

C:\Users\sadaf\AppData\Local\Temp\ipykernel_33664\3445162665.py:108: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_33664\3445162665.py:108: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')


## POT RD, In-Situ, Stationary

In this section, 
1. Constant thresholds, POT of RD, and associated NTR and RF in a 2.5 days windows are achieved.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob
from datetime import timedelta
import os

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'dateTime'

# Name of variable column in each dataset
ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
events_per_year = 5

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:

    base_path = core_directory / Path(f'{loc}')
    decluster_window = timedelta(days=2.5)
    
    output_dir = rf'{base_path}/Stationary'
    
    # Load datasets
    df_rf = pd.read_csv(rf'{dataset_directory}/In_Situ/GHCN_Rainfall_1950_2050_hourly_accumulated.csv')
    df_rf[time_col_rf] = pd.to_datetime(df_rf[time_col_rf], errors='coerce')
    df_rf = df_rf.set_index(time_col_rf)[acc_hr].dropna().sort_index()

    df_ntr = pd.read_csv(rf'{dataset_directory}/In_Situ/NOAA_Tide_Gauge_MSL_All_Predicted.csv')
    df_ntr[time_col_ntr] = pd.to_datetime(df_ntr[time_col_ntr], errors='coerce')
    df_ntr = df_ntr.set_index(time_col_ntr).sort_index()

    df_rd = pd.read_csv(rf'{dataset_directory}/In_Situ/usgs_discharge.csv')
    df_rd['time'] = pd.to_datetime(df_rd[time_col_rd], errors='coerce')
    df_rd = df_rd.drop(columns=[time_col_rd]).set_index('time').sort_index()

    rainfall_series = df_rf
    rd_series = df_rd[rd_col].dropna()

    # ────────────────────────────────────────────────────────────────
    # Stationary POT threshold selection
    # ────────────────────────────────────────────────────────────────
    n_years = rd_series.index.year.nunique()
    thresholds = np.linspace(rd_series.quantile(0.90), rd_series.max(), 50)
    pot_counts = []

    for th in thresholds:
        temp_series = rd_series[rd_series >= th].copy()
        pot_events = []
        while not temp_series.empty:
            idx_max = temp_series.idxmax()
            pot_events.append(idx_max)
            temp_series = temp_series[(temp_series.index < idx_max - decluster_window) | (temp_series.index > idx_max + decluster_window)]
        pot_counts.append(len(pot_events))

    events_per_year_target = np.array(pot_counts) / n_years
    best_idx = np.argmin(np.abs(events_per_year - events_per_year_target))
    best_threshold = thresholds[best_idx]
    print(f"Best threshold: {best_threshold:.2f} gives ~{events_per_year_target[best_idx]:.2f} events/year")
    
    # ────────────────────────────────────────────────────────────────
    # Decluster using best threshold and collect max RF + NTR
    # ────────────────────────────────────────────────────────────────
    rd_above_th = rd_series[rd_series >= best_threshold]
    selected_events = []
    
    while not rd_above_th.empty:
        idx_max = rd_above_th.idxmax()
        selected_events.append(idx_max)
        rd_above_th = rd_above_th[(rd_above_th.index < idx_max - decluster_window) | (rd_above_th.index > idx_max + decluster_window)]
    
    all_pot_stationary = []
    
    for t in selected_events:
        rd_val = rd_series.loc[t]
        
        # NTR window
        ntr_window = df_ntr.loc[t - decluster_window: t + decluster_window]
        max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan
    
        # RF window
        rf_window = rainfall_series.loc[t - decluster_window: t + decluster_window]
        max_rf = rf_window.max() if not rf_window.empty else np.nan
    
        all_pot_stationary.append({
            'time': t,
            'POT_RD': rd_val,
            'Max_NTR_in_5d': max_ntr,
            'Max_RF_in_5d': max_rf
        })
    
    # ────────────────────────────────────────────────────────────────
    # Save final stationary POT results
    # ────────────────────────────────────────────────────────────────
    df_pot_stationary = pd.DataFrame(all_pot_stationary).sort_values('time')
    df_pot_stationary = df_pot_stationary.dropna()
    df_pot_stationary.to_csv(os.path.join(output_dir, f'POT_RD_Stationary_NTR_RF_{events_per_year}_Threshold_{best_threshold:.2f}.csv'), index=False)
    df_pot_stationary.to_csv(os.path.join(output_dir, f'POT_RD_Stationary_NTR_RF_{events_per_year}.csv'), index=False)


OceanSprings RD points: 26578 RD range: 0.031148531 279.2041074
Best threshold: 34.39 gives ~5.24 events/year
CorpusChristi RD points: 27582 RD range: 0.000566337 3539.605824
Best threshold: 93.65 gives ~4.57 events/year


## POT RD, Reanalysis, Time Varying

In [ ]:
# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'valid_time'

# Name of variable column in each dataset
ntr_col = 'Average'
rd_col = 'basin_avg_dis24_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
events_per_year = 10

locations = ['OceanSprings']

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:
    # ────────────────────────────────────────────────────────────────────────────────
    # 📌 Load datasets
    # ────────────────────────────────────────────────────────────────────────────────
    df_rf = pd.read_csv(rf'{dataset_directory}\Reanalysis\APCP_Basin_Average_{loc}_Combined_accumulated.csv')
    df_rf[time_col_rf] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index(time_col_rf)
    df_rf = df_rf[acc_hr]

    df_ntr = pd.read_csv(rf'{dataset_directory}\Reanalysis\Compare_Points_{loc}_Avg.csv')
    df_ntr[time_col_ntr] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index(time_col_ntr)

    df_rd = pd.read_csv(rf'{dataset_directory}\Reanalysis\GloFAS_{loc}_Basin_Avg.csv')
    # df_rd = df_rd.drop(columns=['value'])
    df_rd['time'] = pd.to_datetime(df_rd[time_col_rd])
    df_rd = df_rd.drop(columns=[time_col_rd])
    df_rd = df_rd.set_index('time')
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    rainfall_series = df_rf.dropna()
    decluster_window = timedelta(days=3)
    
    output_dir = rf'{core_directory}\{loc}\Reanalysis\TimeVarying'
    
    rd_series = df_rd[rd_col].dropna()
    # ────────────────────────────────────────────────────────────────────────────────
    # Storage
    # ────────────────────────────────────────────────────────────────────────────────
    all_pot = []
    thresholds = []
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Yearly POT extraction + NTR matching
    # ────────────────────────────────────────────────────────────────────────────────
    for year, group in rd_series.groupby(rd_series.index.year):
        group_sorted = group.sort_values(ascending=False)
        selected = []
    
        for idx, val in group_sorted.items():
            if any(abs((idx - sel).days) < decluster_window.days for sel in selected):
                continue
            selected.append(idx)
            if len(selected) == events_per_year:
                break
    
        if len(selected) < events_per_year:
            continue
    
        # Minimum of selected events becomes the threshold for that year
        pot_values = [rd_series.loc[t] for t in selected]
        threshold_value = min(pot_values)
        thresholds.append({'Year': year, 'Threshold': threshold_value})
    
        # Match with NTR
        for t, val in zip(selected, pot_values):
            start = t - decluster_window
            end = t + decluster_window
            ntr_window = df_ntr.loc[start:end]
            max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan
            all_pot.append({'time': t, 'POT_RD': val, 'Max_NTR_in_5d': max_ntr, 'Year': year})
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Save outputs
    # ────────────────────────────────────────────────────────────────────────────────
    df_pot = pd.DataFrame(all_pot).sort_values('time').reset_index(drop=True)
    df_thresh = pd.DataFrame(thresholds)
    
    df_pot.to_csv(os.path.join(output_dir, f'POT_RD_TimeVarying_{events_per_year}.csv'), index=False)
    df_thresh.to_csv(os.path.join(output_dir, f'RD_TimeVarying_Thresholds_{events_per_year}.csv'), index=False)
    
    df_pot = pd.read_csv(os.path.join(output_dir, f'POT_RD_TimeVarying_{events_per_year}.csv'))
    df_pot = df_pot.dropna()
    df_pot = df_pot.drop(columns=['Year'])
    df_pot['time'] = pd.to_datetime(df_pot['time'])
    df_pot = df_pot.set_index('time')
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in df_pot.index:
        # event_date = pd.to_datetime(event_time)  # usually not necessary if already Timestamp
    
        start_date = event_time - timedelta(days=3)
        end_date = event_time + timedelta(days=3)
    
        # Subset discharge data in that window
        window_data = df_rf.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_value = window_data.max()
        max_rd_series = pd.Series([max_value], name=event_time)
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    df_pot.index = pd.to_datetime(df_pot.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    df_pot = df_pot.copy()
    df_pot = df_pot.reset_index().rename(columns={'time': 'date'})
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  # 'date' is of type datetime.date
    
    # Step 3: Merge on 'date'
    merged_df = df_pot.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.rename(columns={'date':'time'})
    # merged_df = merged_df.drop(columns=['time'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={0:'Max_RF_in_5d'})
    merged_df['time'] = pd.to_datetime(merged_df['time'])
                   
    merged_df.to_csv(rf'{output_dir}\POT_RD_TimeVarying_NTR_RF_{events_per_year}.csv',index=False)      

C:\Users\sadaf\AppData\Local\Temp\ipykernel_63176\3898493563.py:105: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_63176\3898493563.py:105: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')


## POT RD, Reanalysis, Stationary

In [ ]:
# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'valid_time'

# Name of variable column in each dataset
ntr_col = 'Average'
rd_col = 'basin_avg_dis24_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
events_per_year = 10

locations = ['OceanSprings']

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:
    # ────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────
    decluster_window = timedelta(days=2.5)

    output_dir = Path(core_directory) / loc / "Reanalysis" / "Stationary"
    output_dir.mkdir(parents=True, exist_ok=True)

    # ────────────────────────────────────────────────────────────────
    # Load datasets
    # ────────────────────────────────────────────────────────────────
    df_rf = pd.read_csv(Path(dataset_directory) / "Reanalysis" / f"APCP_Basin_Average_{loc}_Combined_accumulated.csv")
    df_rf[time_col_rf] = pd.to_datetime(df_rf[time_col_rf], errors="coerce")
    df_rf = df_rf.set_index(time_col_rf).sort_index()
    df_rf = df_rf[acc_hr]  # Series

    df_ntr = pd.read_csv(Path(dataset_directory) / "Reanalysis" / f"Compare_Points_{loc}_Avg.csv")
    df_ntr[time_col_ntr] = pd.to_datetime(df_ntr[time_col_ntr], errors="coerce")
    df_ntr = df_ntr.set_index(time_col_ntr).sort_index()

    df_rd = pd.read_csv(Path(dataset_directory) / "Reanalysis" / f"GloFAS_{loc}_Basin_Avg.csv")
    df_rd["time"] = pd.to_datetime(df_rd[time_col_rd], errors="coerce")
    df_rd = df_rd.drop(columns=[time_col_rd]).set_index("time").sort_index()

    rainfall_series = df_rf.dropna()
    rd_series = df_rd[rd_col].dropna()

    # ────────────────────────────────────────────────────────────────
    # Stationary POT threshold selection
    # ────────────────────────────────────────────────────────────────
    n_years = rd_series.index.year.nunique()
    thresholds = np.linspace(rd_series.quantile(0.90), rd_series.max(), 50)
    pot_counts = []

    for th in thresholds:
        temp_series = rd_series[rd_series >= th].copy()
        pot_events = []
        while not temp_series.empty:
            idx_max = temp_series.idxmax()
            pot_events.append(idx_max)
            temp_series = temp_series[
                (temp_series.index < idx_max - decluster_window) |
                (temp_series.index > idx_max + decluster_window)
            ]
        pot_counts.append(len(pot_events))

    events_per_year_target = np.array(pot_counts) / n_years
    best_idx = np.argmin(np.abs(events_per_year - events_per_year_target))
    best_threshold = thresholds[best_idx]
    print(f"{loc} | Best threshold: {best_threshold:.2f} gives ~{events_per_year_target[best_idx]:.2f} events/year")

    # ────────────────────────────────────────────────────────────────
    # Decluster using best threshold and collect max RF + NTR
    # ────────────────────────────────────────────────────────────────
    rd_above_th = rd_series[rd_series >= best_threshold].copy()
    selected_events = []

    while not rd_above_th.empty:
        idx_max = rd_above_th.idxmax()
        selected_events.append(idx_max)
        rd_above_th = rd_above_th[
            (rd_above_th.index < idx_max - decluster_window) |
            (rd_above_th.index > idx_max + decluster_window)
        ]

    all_pot_stationary = []

    for t in selected_events:
        rd_val = rd_series.loc[t]

        ntr_window = df_ntr.loc[t - decluster_window: t + decluster_window]
        max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan

        rf_window = rainfall_series.loc[t - decluster_window: t + decluster_window]
        max_rf = rf_window.max() if not rf_window.empty else np.nan

        all_pot_stationary.append({
            "time": t,
            "POT_RD": rd_val,
            "Max_NTR_in_5d": max_ntr,
            "Max_RF_in_5d": max_rf
        })

    # ────────────────────────────────────────────────────────────────
    # Save
    # ────────────────────────────────────────────────────────────────
    df_pot_stationary = pd.DataFrame(all_pot_stationary).sort_values("time").dropna()

    df_pot_stationary.to_csv(
        output_dir / f"POT_RD_Stationary_NTR_RF_{events_per_year}_Threshold_{best_threshold:.2f}.csv",
        index=False
    )
    df_pot_stationary.to_csv(
        output_dir / f"POT_RD_Stationary_NTR_RF_{events_per_year}.csv",
        index=False
    )


OceanSprings RD points: 16953 RD range: 4.144808 307.33337
OceanSprings | Best threshold: 86.89 gives ~10.26 events/year
CorpusChristi RD points: 16953 RD range: 2.6524658 166.02982
CorpusChristi | Best threshold: 47.43 gives ~10.21 events/year
